## ② IQL —— **退出训练主线(D69 用户决议)**
IQL 结构性 ≈ BC(D68.1), 降级为横向对比对象;**本格暂不运行**, 等在线训练完成后做全家福对比时再跑。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## ① 生成离线数据(c12 口径; 本地已生成+拷贝则跳过)
本地: `python scripts/33_build_offline_dataset.py train 24 6 5000 0 6 12` → 拷 `artifacts/dataset/offline_shards_train_c12/`。

In [ ]:
!{PREFIX} scripts/33_build_offline_dataset.py train 24 70 5000 0 6 12   # 客运口径 c12, 无 game-over -> offline_shards_train_c12


## ② IQL —— **退出训练主线(D69 用户决议)**
IQL 结构性 ≈ BC(D68.1), 降级为横向对比对象;**本格暂不运行**, 等在线训练完成后做全家福对比时再跑。

In [ ]:
!{PREFIX} scripts/34_train_iql.py \
    --dataset artifacts/dataset/offline_shards_train_c12 \
    --shard-glob 'signaller_*.pkl' \
    --control-classes 12 \
    --encoder gnn --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --iters 3000 --batch 128 --eval-every 150 --gamma 0.99 --tau 0.7 --beta 0.5 \
    --run-tag iql_sig_s0 --resume


## ③ PPO 并行消融(D69): 8192+时机锚(后台) vs 8192 无锚(前台)
共同配置: 升档=**完成率1.0 且 延误≤1.0×fcfs**(档内同日) + best-rescue + 8192批量 + lr 1e-4。差异只有 `--bc-anchor-beta 0.02`。**先跑 ③a(后台启动), 再跑 ③b(前台, 看实时输出)**; ③c 随时看后台日志。72 核跑两份没问题。

### ③a 后台: ppo_8192_anchor_s0(时机锚 β=0.02)

In [ ]:
import subprocess, shlex, os
log = f'{PROJ}/artifacts/reports/ppo_8192_anchor_s0.log'
cmd = (f'{PREFIX} scripts/13_train_ppo.py \
    --bc-anchor-beta 0.02 --run-tag ppo_8192_anchor_s0 \
    --encoder gnn --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt  --control-classes 12  --kl-anchor schedule --kl-beta 0.03 --seed 0 --reserve-window wide  --curriculum --curriculum-tiers 185,286,300,999999  --promote-at 1.0 --promote-delay-k 1.0  --train-eval-days 6 --iters 120 --steps-per-iter 8192 --lr 1e-4 --eval-every 5  --early-stop-patience 8 --best-rescue --resume')
subprocess.Popen(['bash','-c',f'nohup {cmd} > {log} 2>&1 &'])
print('后台已启动(时机锚 β=0.02) → 日志:', log)


### ③b 前台: ppo_8192_s0(无锚)

In [ ]:
!{PREFIX} scripts/13_train_ppo.py \
    --run-tag ppo_8192_s0 \
    --encoder gnn --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --control-classes 12 \
    --kl-anchor schedule --kl-beta 0.03 --seed 0 --reserve-window wide \
    --curriculum --curriculum-tiers 185,286,300,999999 \
    --promote-at 1.0 --promote-delay-k 1.0 \
    --train-eval-days 6 --iters 120 --steps-per-iter 8192 --lr 1e-4 --eval-every 5 \
    --early-stop-patience 8 --best-rescue --resume


### ③c 看后台日志(随时重跑本格)

In [ ]:
!tail -n 25 {PROJ}/artifacts/reports/ppo_8192_anchor_s0.log


## ④ 对比(主指标=延误)

In [ ]:
import json, torch
from pathlib import Path
rep=Path('artifacts/reports')
def load(tag):
    if (rep/f'{tag}.json').exists(): return json.loads((rep/f'{tag}.json').read_text())
    if (rep/f'{tag}_last.pt').exists(): return torch.load(rep/f'{tag}_last.pt',map_location='cpu',weights_only=False)
    return None
runs=[('8192 无锚','ppo_8192_s0'),('8192+时机锚','ppo_8192_anchor_s0'),('旧2048','ppo_p12_s0')]
loaded={n:load(t) for n,t in runs}
base=next((d.get('baselines') for d in loaded.values() if d and d.get('baselines')),{})
print(f"{'run':<12}{'best完成':>9}{'best延误':>11}{'best_rew':>10}{'@it':>5}")
for k in ['fcfs','lookup_most_frequent','greedy_set','signaller_replay']:
    if k in base:
        b=base[k]; print(f"{k:<12}{b['eval_completion_rate']:>9.3f}{b['eval_weighted_secondary_delay_s']:>11.0f}{b['eval_episode_reward']:>10.0f}{'':>5}")
for name,_ in runs:
    d=loaded.get(name)
    if not d or not d.get('best'): print(f'{name:<12}(missing)'); continue
    b=d['best']; print(f"{name:<12}{b.get('completion',0):>9.3f}{b.get('delay',0):>11.0f}{b.get('reward',0):>10.0f}{b.get('iter',-1):>5}")
try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,2,figsize=(11,3.2))
    for name,_ in runs:
        d=loaded.get(name)
        if d and d.get('evals'):
            ev=d['evals']
            axes[0].plot([e['iter'] for e in ev],[e['eval_completion_rate'] for e in ev],marker='.',label=name)
            axes[1].plot([e['iter'] for e in ev],[e['eval_weighted_secondary_delay_s'] for e in ev],marker='.',label=name)
    axes[0].axhline(1.0,ls='--',c='g',lw=.8); axes[0].set_title('val 完成率'); axes[0].legend(fontsize=8)
    if 'fcfs' in base: axes[1].axhline(base['fcfs']['eval_weighted_secondary_delay_s'],ls='--',c='b',lw=.8,label='fcfs')
    if 'greedy_set' in base: axes[1].axhline(base['greedy_set']['eval_weighted_secondary_delay_s'],ls='--',c='r',lw=.8,label='always-set')
    axes[1].set_yscale('log'); axes[1].set_title('val 延误(log)'); axes[1].legend(fontsize=7)
    plt.tight_layout(); plt.savefig('artifacts/reports/d69_compare.png',dpi=110)
    print('fig -> artifacts/reports/d69_compare.png')
except Exception as e: print('plot skip',e)


## 判读
- **PPO 完成率上不去 1.0** → 发我 EVAL 行, 查(不该发生, 傻瓜策略都能 1.0)。
- **延误压过 greedy_set(always-set)** → RL 学到了超越「立即放行」的时机/选路 → 成功故事。
- **延误只打平 always-set** → 说明该口径下「立即放行」近似最优 → 也是干净结论(报告它, 转向全控难度)。
- test 封存(D43)。